In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp
from delta.tables import DeltaTable

# Base path do landing zone
BASE_PATH = "abfss://landing@saaulafatecdatalake.dfs.core.windows.net/fatec_store/"

# Lista de tabelas
tabelas = [
    "avaliacoes",
    "categorias",
    "clientes",
    "enderecos_clientes",
    "entregas",
    "itens_pedido",
    "pagamentos",
    "pedidos",
    "produtos",
    "produtos_promocoes",
    "promocoes",
    "subcategorias",
    "vendedores"
]

# Database bronze
spark.sql("CREATE DATABASE IF NOT EXISTS bronze")

for tabela in tabelas:

    print(f"Lendo tabela: {tabela}")

    # Caminho do parquet no landing
    parquet_path = f"{BASE_PATH}{tabela}/{tabela}.parquet"

    # Leitura parquet
    df = spark.read.parquet(parquet_path)

    # Adiciona metadata de ingestão
    df = df.withColumn("ingestion_timestamp", current_timestamp())

    # Nome tabela bronze
    bronze_table = f"bronze.{tabela}"

    print(f"Salvando tabela Delta: {bronze_table}")

    # Salva como Delta Table
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(bronze_table)
    )

    print(f"Tabela {bronze_table} criada com sucesso")


print("Carga Bronze finalizada.")